# Merge the LoRA adapter and evaluate on the test split
Run on a **fresh Colab T4 server**, top to bottom. Needs a Hugging Face token (Read access is enough) to download the private adapter repo.

Plan: merge adapter into fp16 base, sanity-check 5 examples, run the full test set, save results to the Hub (Colab's disk is wiped on disconnect).

## 1. Setup

In [ ]:
!nvidia-smi -L
!git clone https://github.com/hannna24/Github-IssueHawk.git
%cd Github-IssueHawk/training
!pip install -q -U transformers peft accelerate python-dotenv scikit-learn

## 2. Log in to Hugging Face
Paste your token when prompted (it is not stored in this notebook).

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('Hugging Face token: '))

## 3. Merge the adapter into the base model
Downloads Qwen2.5-3B-Instruct in fp16 (~6 GB), attaches `final-adapter/` from your Hub repo, merges, and saves to `../checkpoints/merged`. A few minutes.

In [ ]:
!python merge_adapter.py --adapter hanabakeer24/issuehawk-qwen2.5-3b-lora --subfolder final-adapter --out ../checkpoints/merged

## 4. Sanity check (5 issues)
Expect `status counts: {'ok': 5}` and raw outputs like `{"label": "bug"}`. If you see extra text or `malformed_json`, stop here. Writes no result files.

In [ ]:
!python predict_student.py --model ../checkpoints/merged --limit 5

## 5. Full test-set evaluation
Runs all test issues, prints coverage, accuracy and macro-F1, and writes `eval/results/predictions_student.jsonl` and `latest_student.json`.

In [ ]:
!python predict_student.py --model ../checkpoints/merged

## 6. Per-class results

In [ ]:
import json
r = json.load(open('../eval/results/latest_student.json'))
for k in r['labels']:
    v = r['report'][k]
    print(f"{k:16s} precision {v['precision']:.3f}  recall {v['recall']:.3f}  f1 {v['f1-score']:.3f}  support {int(v['support'])}")
print('accuracy', round(r['report']['accuracy'], 4), ' macro F1', round(r['report']['macro avg']['f1-score'], 4), ' n', r['n'])
print('confusion matrix (rows=true, cols=pred):', r['labels'])
for row in r['confusion_matrix']:
    print(row)

## 7. Save the results to the Hub *(Colab's disk is wiped on disconnect)*
Uploads the two result files to an `eval/` folder in your adapter repo.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
for name in ('predictions_student.jsonl', 'latest_student.json'):
    api.upload_file(
        path_or_fileobj=f'../eval/results/{name}',
        path_in_repo=f'eval/{name}',
        repo_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',
    )
print('uploaded')